In [ ]:
import json
import os

import agama
import gizmo_analysis as gizmo
import matplotlib.pyplot as plt
import numpy as np

agama.setUnits(mass=1, length=1, velocity=1)
units = agama.getUnits()


# plt.rc("font", family="Nimbus Roman")
plt.rc("font", family="Nimbus Roman No9 L")

SMALL_SIZE = 10
MEDIUM_SIZE = 10
BIGGER_SIZE = 10

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title


In [ ]:
sim_dir = "/Users/z5114326/Documents/simulations/"
# sim_dir = "/Volumes/Expansion/simulations/"
sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]

sim_codes = sim_dir + "simulation_codes.json"
with open(sim_codes) as sim_json:
    sim_data = json.load(sim_json)

scmap = plt.get_cmap("tab10")
colors = [scmap(i) for i in np.linspace(0, 1, len(sim_lst))]
sim_colors = {sim: colors[i] for i, sim in enumerate(sim_lst)}

In [ ]:
def convert(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, dict):
        return {k: convert(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert(v) for v in obj]
    return obj

In [ ]:
rlim = 20  # kpc
snap = 600

eps_path = "../data/sim_data/circularity.json"
if os.path.exists(eps_path):
    print("Loading existing gal_dict JSON...")
    with open(eps_path, "r", encoding="utf-8") as f:
        dat_dict = json.load(f)

else:
    dat_dict = {sim: {} for sim in sim_lst}

    for sim in sim_lst:
        fir_dir = sim_dir + sim + "/" + sim + "_res7100"

        part = gizmo.io.Read.read_snapshots(
            ["star"],
            "index",
            snap,
            fir_dir,
            assign_hosts_rotation=True,
            assign_formation_coordinates=True,
        )

        pot_file = sim_dir + sim + "/potentials/snap_%d/combined_snap_%d.ini" % (snap, snap)
        pot_nbody = agama.Potential(pot_file)
        af = agama.ActionFinder(pot_nbody, interp=False)

        rmsk = part["star"].prop("host.principal.distance.total") < rlim
        pxyz = part["star"].prop("host.principal.distance", rmsk)
        vxyz = part["star"].prop("host.principal.velocity", rmsk)
        ages = part["star"].prop("age", rmsk)
        mass = part["star"]["mass"][rmsk]

        posvel = np.column_stack((pxyz, vxyz))

        lz = af(posvel)[:, 2]

        ek = 0.5 * np.sum(np.array(vxyz) ** 2, axis=1)
        ep = pot_nbody.potential(pxyz)
        et = ek + ep

        r_circs = pot_nbody.Rcirc(E=et)
        xyz = np.column_stack((r_circs, r_circs * 0, r_circs * 0))
        v_circs = np.sqrt(-r_circs * pot_nbody.force(xyz)[:, 0])
        vel = np.column_stack((v_circs * 0, v_circs, v_circs * 0))
        init_conds = np.concatenate((xyz, vel), axis=1)
        lz_circ = af(init_conds)[:, 2]

        eps = lz / np.array(lz_circ)

        bin_size = 0.2
        bins = np.arange(0, 13.6 + bin_size, bin_size)
        bin_cens = bins[1:] - bin_size / 2

        eps_16 = np.zeros(len(bin_cens))
        eps_50 = np.zeros(len(bin_cens))
        eps_84 = np.zeros(len(bin_cens))

        for i in range(len(bins[1:])):
            bmin = bins[i]
            bmax = bins[i + 1]

            bmsk = (bmin <= ages) & (ages < bmax)
            eps_16[i] = np.percentile(eps[bmsk], 16, weights=mass[bmsk], method="inverted_cdf")
            eps_50[i] = np.percentile(eps[bmsk], 50, weights=mass[bmsk], method="inverted_cdf")
            eps_84[i] = np.percentile(eps[bmsk], 84, weights=mass[bmsk], method="inverted_cdf")

        dat_dict[sim]["lbt"] = bin_cens
        dat_dict[sim]["eps_16"] = eps_16
        dat_dict[sim]["eps_50"] = eps_50
        dat_dict[sim]["eps_84"] = eps_84

    dat_dict_json = convert(dat_dict)

    with open(eps_path, "w") as f:
        json.dump(dat_dict_json, f, indent=4)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))

for sim in sim_lst:
    ax.plot(dat_dict[sim]["lbt"], dat_dict[sim]["eps_50"], c=sim_colors[sim], label=sim, lw=2)

ax.axhline(0.2, lw=0.5, c="grey")
ax.axhline(0.8, lw=0.5, c="grey")

ax.set_xlim(13.6, 0)
ax.set_ylim(-0.2, 1.0)

ax.set_xlabel(r"Lookback Time (Gyr)")
ax.set_ylabel(r"$j_{z}$ / $j_{z}(E)$")

ax.legend(loc="upper left")